# Stage 1: reproduce the metadata coverage audit

One frozen forecast-v4 score per wallet; coverage grain is a distinct wallet-condition
pair. This notebook verifies the archived inputs and reproduces the complete audit.
It does not fetch present-day observations or publish production scores.

Set `MSOS_ATTRITION_SNAPSHOT`, `MSOS_METADATA_DATA_DIR`, and `MSOS_ACTIVITY_DATASET`
to the canonical frozen scores, matching markets/hydration directory, and verified
Parquet activity dataset. Install the ingestor's benchmark extra and use Python 3.12.
The full local replay scans 64 bounded partitions and checks their hashes twice.


In [1]:
import json
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'services/polymarket-ingestor/src').is_dir())
sys.path.insert(0, str(root / 'services/polymarket-ingestor/src'))
from marketsignalos_polymarket.metadata_audit import build_audit

reference = json.loads((root / 'docs/benchmarks/2026-09-10-metadata-coverage.json').read_text())
snapshot = Path(os.environ['MSOS_ATTRITION_SNAPSHOT'])
data_dir = Path(os.environ['MSOS_METADATA_DATA_DIR'])
dataset = Path(os.environ['MSOS_ACTIVITY_DATASET'])
benchmark = root / 'docs/benchmarks/2026-09-08-enrichment-shadow.json'


In [2]:
report, replays = build_audit(snapshot, benchmark, data_dir, dataset, memory_mb=512)
for key in ('summary', 'before_waterfall', 'after_waterfall', 'after_counterfactuals', 'wallets'):
    assert report[key] == reference[key], key
assert report['provenance']['snapshot'] == reference['provenance']['snapshot']
assert report['provenance']['audit_source_sha256'] == reference['provenance']['audit_source_sha256']
print('Full audit reproduced: summary, both waterfalls, counterfactuals, and all per-wallet counts.')
print('Verified Parquet files:', report['provenance']['verified_parquet_files'])
print('Frozen score SHA-256:', report['provenance']['snapshot']['sha256'])


Full audit reproduced: summary, both waterfalls, counterfactuals, and all per-wallet counts.
Verified Parquet files: 65
Frozen score SHA-256: 01564d095e7945cee89895f3f278a0f0d66e9aee6b88530325f27e10a447dc5f


In [3]:
print(json.dumps(report['summary'], indent=2))
print(json.dumps([{key: wallet[key] for key in ('wallet', 'stored', 'recomputed')}
                  for wallet in report['wallets'] if wallet['metadata_only_candidate']], indent=2))


{
  "wallets": 833,
  "hydration_count_mismatches": 537,
  "wallet_condition_pairs": 758510,
  "condition_reason_counts": {
    "covered": 737950,
    "present_unsettled": 20352,
    "missing_market_rows": 208
  },
  "metadata_failures_before": 701,
  "metadata_failures_after": 592,
  "trusted_before": 125,
  "trusted_after": 226,
  "tailable_before": 0,
  "tailable_after": 0
}
[
  {
    "wallet": "0x521070c99db06e54af5e8e4a91d6858decdfbd53",
    "stored": {
      "metadata_condition_count": 21491,
      "metadata_covered_count": 8364,
      "metadata_coverage": 0.38918617095528363
    },
    "recomputed": {
      "conditions": 21491,
      "covered": 20221,
      "missing_market_rows": 36,
      "present_unsettled": 1234,
      "ratio": 0.9409054953236239
    }
  },
  {
    "wallet": "0x5a218c7ad04135830a45c41aaed7294df7809318",
    "stored": {
      "metadata_condition_count": 17472,
      "metadata_covered_count": 11905,
      "metadata_coverage": 0.6813759157509157
    },
    "reco

## Interpretation

Saved coverage disagrees with the score inputs for 537 wallets. Recomputing the
existing rule yields 592 metadata failures and 226 data-trusted wallets, with no
qualified wallets. Most remaining uncovered pairs have market rows but do not
satisfy the legacy settlement proxy. Missing-row fetch causes are unknown because
these files have no per-condition attempt receipts. No numeric model output or
performance threshold was changed in the diagnostic replay. See the report and
runbook for scope, remaining work, and the $15/month deployment constraint.
